# L61 · nn.Module 实战 — Linear / ReLU / Sequential，参数管理与模型保存

**目标**：用 `nn.Module` 复现 `L57` 中手写的 MLP，使两者在同一数据上参数数量和前向输出形状完全一致。

🔗 **Aurora 连接**：Month 2 关键词识别模型（KWS）、Month 3 ASR 微调、Month 5 LLM 都以 `nn.Module` 为骨架——掌握它的注册机制和序列化（serialization）接口是后续一切工作的前提。

`nn.Module` 解决的核心问题是**参数所有权**：把权重张量（tensor）绑定到模型对象，而不是散落在函数局部变量里，从而让优化器、序列化和设备迁移都能自动找到所有参数。
只要把子层赋值为实例属性（`self.fc = nn.Linear(...)`），PyTorch 就会自动注册其参数，`parameters()` 和 `state_dict()` 无需额外声明。
与手写 MLP 相比，`nn.Module` 不改变数学，只改变**参数的生命周期管理**。

In [ ]:
import torch
import torch.nn as nn

## 1. `forward()` 与参数自动注册

`nn.Module` 要求子类实现 `forward()`，调用时写 `model(x)` 而非 `model.forward(x)`——
前者会触发钩子（hooks），后者绕过它们。

参数注册有两种方式：
- 赋值内置层（`nn.Linear`, `nn.Conv1d` …）：参数自动注册
- 赋值裸张量：**必须** 用 `nn.Parameter(tensor)` 包装，否则不会被追踪

```python
# 正确：参数被注册
self.w = nn.Parameter(torch.randn(10, 5))
# 错误：不被追踪，优化器看不到它
self.w = torch.randn(10, 5)
```

In [ ]:
# 演示：nn.Parameter 注册
class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(4, 2)           # 自动注册 weight + bias
        self.scale = nn.Parameter(torch.ones(1))  # 手动注册
        self.buf = torch.zeros(1)            # 未注册，仅普通张量

    def forward(self, x):
        return self.fc(x) * self.scale

tiny = TinyNet()
param_names = [n for n, _ in tiny.named_parameters()]
print('注册的参数:', param_names)  # fc.weight, fc.bias, scale
print('buf 未出现:', 'buf' not in param_names)

### `register_buffer()` — 正确的非参数状态注册方式

`buf` 的反例展示了**不注册**的后果；正确做法是用 `register_buffer()`，适用于不需要梯度但需要随模型移动（`.to(device)`）并出现在 `state_dict()` 中的张量（如 BatchNorm 的 running mean）。

> 区别一览：
>
> | 方式 | `parameters()` | `state_dict()` | 梯度 |
> |---|---|---|---|
> | `nn.Parameter` | ✅ | ✅ | ✅ |
> | `register_buffer()` | ❌ | ✅ | ❌ |
> | 裸张量属性 | ❌ | ❌ | ❌ |


In [ ]:
# 演示：register_buffer — 出现在 state_dict 但不在 parameters()
class TinyNetV2(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(4, 2)
        self.scale = nn.Parameter(torch.ones(1))   # 有梯度，出现在 parameters()
        self.register_buffer('running_mean', torch.zeros(1))  # 无梯度，出现在 state_dict
        self.plain_buf = torch.zeros(1)             # 不注册，state_dict 中不出现

    def forward(self, x):
        return self.fc(x) * self.scale

tiny2 = TinyNetV2()
print('state_dict keys:', list(tiny2.state_dict().keys()))
# 预期: fc.weight, fc.bias, scale, running_mean  (不含 plain_buf)
assert 'running_mean' in tiny2.state_dict(), 'register_buffer 的张量应出现在 state_dict'
assert 'plain_buf' not in tiny2.state_dict(), '裸张量不出现在 state_dict'
assert 'running_mean' not in [n for n, _ in tiny2.named_parameters()], 'buffer 不计入梯度参数'
print('✅ register_buffer 验证通过：running_mean 在 state_dict 中，不在 parameters() 中')


## 2. `named_parameters()` 与 `state_dict()`

`named_parameters()` 递归遍历所有子模块，返回 `(名称, 张量)` 对；常用于统计参数量或做分层学习率。

```python
total = sum(p.numel() for p in model.parameters())
```

`state_dict()` 返回 `OrderedDict`，键为参数名，值为张量——这是保存/加载检查点（checkpoint，ckpt）的标准接口：

```python
torch.save(model.state_dict(), 'ckpt.pt')
model.load_state_dict(torch.load('ckpt.pt'))
```

两个模型结构相同时，可直接把 `state_dict` 从一个传给另一个，实现权重复制。

In [ ]:
# 演示：state_dict 键名与形状
for name, param in tiny.named_parameters():
    print(f'{name:20s}  shape={list(param.shape)}  numel={param.numel()}')

print('\nstate_dict keys:', list(tiny.state_dict().keys()))

## 3. `nn.Sequential`：省掉 `forward()`

`nn.Sequential` 按顺序串联子模块，自动实现 `forward(x) = layer_n(... layer_1(x))`，无需手写循环。子模块以整数索引命名（`0`, `1`, `2` …），也可用 `OrderedDict` 给定字符串名。

```python
net = nn.Sequential(
    nn.Linear(40, 64),
    nn.ReLU(),
    nn.Linear(64, 10),
)
# 等价于：x -> Linear -> ReLU -> Linear -> 输出
```

适用条件：层之间**无分支、无跳连（skip connection）**。有残差连接（residual connection）时必须回到手写 `forward()`。

In [ ]:
# 演示：nn.Sequential 前向
seq = nn.Sequential(
    nn.Linear(40, 64),
    nn.ReLU(),
    nn.Linear(64, 10),
)
x_demo = torch.randn(8, 40)
print('Sequential 输出形状:', seq(x_demo).shape)  # (8, 10)
print('子模块列表:', list(seq.children()))

## 4. ✏️ 实现 `class AudioMLP(nn.Module)`

**推理路线**：
1. `__init__` 中用 `nn.Sequential` 串联 `nn.Linear(in_features, hidden)` → `nn.ReLU()` → `nn.Linear(hidden, out_features)`，赋值给 `self.layers`。
2. `forward(self, x)` 直接 `return self.layers(x)`，参数形状由子模块自动处理。
3. 调用 `model(x)` 触发 `__call__` → `forward`，不要直接调 `forward()`。

**参考输入输出**：
```
m = AudioMLP(40, 64, 10)
x = torch.randn(8, 40)   # batch=8, in_features=40
y = m(x)                  # 期望 shape: (8, 10)
```

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


In [ ]:
class AudioMLP(nn.Module):
    def __init__(self, in_features, hidden, out_features):
        super().__init__()
        # ✏️ TODO: 定义 self.layers = nn.Sequential(...)
        raise NotImplementedError("TODO: 在 __init__ 中定义 self.layers = nn.Sequential(nn.Linear(in_features, hidden), nn.ReLU(), nn.Linear(hidden, out_features))")

    def forward(self, x):
        # ✏️ TODO: return self.layers(x)
        raise NotImplementedError("TODO: 在 forward 中返回 self.layers(x)")


In [ ]:
try:
    m = AudioMLP(40, 64, 10)
    out = m(torch.randn(8, 40))
    assert out.shape == (8, 10), f'期望 (8,10)，得到 {out.shape}'
    print('✅ AudioMLP 前向输出形状正确:', out.shape)
except NotImplementedError as e:
    print(f'⚠️ 练习未完成，请先实现 AudioMLP：{e}')


## 5. 参数实验：验证参数总量

对 `AudioMLP(40, 64, 10)` 手算参数数量：

```
layers.0 (Linear 40→64): weight=40×64=2560, bias=64     → 2624
layers.2 (Linear 64→10): weight=64×10=640,  bias=10     →  650
合计: 2624 + 650 = 3274
```

**预期现象**：`sum(p.numel() for p in m.parameters())` 返回 `3274`，与手写 `L57` 的同构网络完全一致（若手写版有偏置的话）。

修改 `hidden=128` 后总量变为 `40×128+128 + 128×10+10 = 5248 + 1290 = 6538`，可自行验证。

In [ ]:
try:
    total = sum(p.numel() for p in m.parameters())
    print(f'参数总量: {total}')
    assert total == 3274, f'期望 3274，得到 {total}'
    print('✅ 参数总量与手算一致')

    # 查看每层参数明细
    for name, p in m.named_parameters():
        print(f'  {name:25s}  {list(p.shape)}  = {p.numel()}')

    # 实验：hidden=128 时参数数量
    m128 = AudioMLP(40, 128, 10)
    print(f'\nhidden=128 时参数总量: {sum(p.numel() for p in m128.parameters())}')
except NotImplementedError as e:
    print(f'⚠️ 练习未完成，请先实现 AudioMLP：{e}')


## 本课收束

`AudioMLP.forward()` 通过 `nn.Sequential` 把线性变换和 ReLU 串联，输出形状 `(batch, out_features)`，与手写 `L57` 的数学等价但参数生命周期由 `nn.Module` 统一管理。
`named_parameters()` 和 `state_dict()` 是 Aurora Month 2 KWS 模型保存检查点、Month 3 ASR 加载预训练权重的直接基础。
下一节（L62）将把特征提取器包装进 `torch.utils.data.Dataset`，构建自定义 `__getitem__`，完成音频数据批量加载。